# Μάθημα 3: Επεξεργασία Φυσικής Γλώσσας (NLP)

Οι υπολογιστές δεν καταλαβαίνουν λέξεις. Καταλαβαίνουν μόνο αριθμούς. Το πρώτο βήμα στο NLP είναι να 'κόψουμε' το κείμενο σε λέξεις ή κομμάτια λέξεων (Tokenization) και μετά να τα κάνουμε μαθηματικά διανύσματα (Embeddings).

## 1. Tokenization και Embeddings
Η πρόταση 'Αγαπώ τον σκύλο μου', κόβεται σε ['Αγαπώ', 'τον', 'σκύλο', 'μου']. Κάθε λέξη παίρνει ένα ID (π.χ. 45, 12, 108, 9). Μετά, το επίπεδο `Embedding` μετατρέπει αυτό το ID σε έναν πίνακα π.χ. 16 αριθμών, που κρύβει το 'νόημα' της λέξης.

In [ ]:
import torch
import torch.nn as nn

# Έστω ένα λεξιλόγιο 1000 λέξεων. Κάθε λέξη θα γίνει ένα διάνυσμα με 16 διαστάσεις (16 αριθμούς).
embedding_layer = nn.Embedding(num_embeddings=1000, embedding_dim=16)

# Μια πρόταση 4 λέξεων (τα IDs τους)
sentence_ids = torch.tensor([[45, 12, 108, 9]])

embedded_sentence = embedding_layer(sentence_ids)
print("Σχήμα του Embedded Text:", embedded_sentence.shape)
# Το αποτέλεσμα (1, 4, 16) σημαίνει: 1 πρόταση, 4 λέξεις, 16 αριθμοί (features) ανά λέξη.

## 2. LSTMs: Διαβάζοντας με 'Μνήμη'
Το πρόβλημα με τα παλιά δίκτυα (RNNs) ήταν το *Vanishing Gradient*: Ξεχνούσαν την αρχή της πρότασης μέχρι να φτάσουν στο τέλος της. Το LSTM (Long Short-Term Memory) φτιάχτηκε για να θυμάται σημαντικές λέξεις για πολλή ώρα.

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_classes):
        super(TextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Το LSTM παίρνει το Embedding και βγάζει ένα Hidden State
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        # Το τελικό επίπεδο αποφασίζει την κατηγορία (π.χ. 2 classes: Positive / Negative)
        self.fc = nn.Linear(hidden_dim, output_classes)

    def forward(self, text):
        embedded = self.embedding(text)
        # Το LSTM επιστρέφει το output όλων των βημάτων και το (hidden, cell) state του τελευταίου βήματος
        _, (hidden, _) = self.lstm(embedded)
        
        # Παίρνουμε το τελευταίο hidden state (αφού διάβασε όλη την πρόταση)
        final_state = hidden[-1]
        
        # Περνάμε το τελικό 'συμπέρασμα' από το Linear layer για ταξινόμηση
        return self.fc(final_state)

nlp_model = TextClassifier(vocab_size=1000, embed_dim=16, hidden_dim=32, output_classes=2)
print("Το μοντέλο μας (Embedding -> LSTM -> Linear):\n", nlp_model)

## 3. Πρόβλεψη (Sentiment Analysis Simulation)
Ας περάσουμε την πρότασή μας μέσα από το μοντέλο για να δούμε αν είναι Θετική (Class 1) ή Αρνητική (Class 0).

In [ ]:
prediction = nlp_model(sentence_ids)

print("Ακατέργαστα αποτελέσματα (Logits) για τις 2 κατηγορίες:", prediction.data)
predicted_class = torch.argmax(prediction).item()
print(f"Το μοντέλο πιστεύει ότι η πρόταση ανήκει στην Κατηγορία: {predicted_class}")